# ACC Reinstatement vs Subject-Level Covariates

Correlate ACC reinstatement with ACC electrode count, total reward, mean ACC feedback HFA, and mean ACC cue HFA.

In [ ]:
import os
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import pearsonr, spearmanr

from plot_config import set_plot_style

set_plot_style()


In [ ]:
# ---- Config ----
reinstatement_metric = 'MeanAccuracy'  # or 'ReinstateEffect'
corr_type = 'spearman'  # 'spearman' or 'pearson'
run_secondary_pearson = True
label_points = True

acc_roi = 'ACC'
hfa_band = 'hfa'
feedback_event = 'feedback_start'
feedback_window: Tuple[float, float] = (0.0, 1.5)
cue_event = 'cue_on'
cue_window: Tuple[float, float] = (0.0, 1.0)

base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
acc_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/'
    'ACC_rpe_category_HFA_subject_reinstatement.csv'
)
hpc_theta_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'HPC_memory_outcome_hit_miss_theta_subject_reinstatement.csv'
)
acc_hfa_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'ACC_memory_outcome_hit_miss_hfa_subject_reinstatement.csv'
)
learn_df_csv = f'{load_dir}/learn_df_RWH.csv'

analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/reinstatement_subject_covariates'
)
figure_output_root = Path(
    '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
)
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

analysis_prefix = f'rpe_category_HFA_{acc_roi}_{reinstatement_metric}_vs_subject_covariates'
paths: Dict[str, Path] = {
    'merged_csv': analysis_output_root / f'{analysis_prefix}_merged_subject_table.csv',
    'results_csv': analysis_output_root / f'{analysis_prefix}_correlation_results.csv',
    'summary_txt': analysis_output_root / f'{analysis_prefix}_summary.txt',
    'scatter_pdf': figure_output_root / 'acc_reinstatement_vs_subject_covariates_scatter.pdf',
    'hpc_merged_csv': analysis_output_root / f'{analysis_prefix}_hpc_theta_acc_hfa_merged_subject_table.csv',
    'hpc_results_csv': analysis_output_root / f'{analysis_prefix}_hpc_theta_acc_hfa_correlation_results.csv',
    'hpc_scatter_pdf': figure_output_root / 'acc_reinstatement_vs_hpc_theta_acc_hfa_scatter.pdf',
}

predictor_order = [
    'ACC_NumElectrodes',
    'total_reward',
    'alpha',
    'beta',
    'ACC_mean_hfa_feedback',
    'ACC_mean_hfa_cue',
]
predictor_labels = {
    'ACC_NumElectrodes': 'ACC electrode count',
    'total_reward': 'Total reward',
    'alpha': 'Alpha',
    'beta': 'Beta',
    'ACC_mean_hfa_feedback': 'Mean ACC HFA during feedback',
    'ACC_mean_hfa_cue': 'Mean ACC HFA during cue',
}

hpc_predictor_order = [
    'HPC_theta_MeanAccuracy',
    'ACC_hfa_MeanAccuracy',
]
hpc_predictor_labels = {
    'HPC_theta_MeanAccuracy': 'HPC theta mean accuracy',
    'ACC_hfa_MeanAccuracy': 'ACC HFA mean accuracy',
}

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subj_ids = subj_df.MSSMCode.tolist()

print('ACC reinstatement CSV:', acc_reinstatement_csv)
print('HPC theta reinstatement CSV:', hpc_theta_reinstatement_csv)
print('ACC HFA reinstatement CSV:', acc_hfa_reinstatement_csv)
print('Behavior CSV:', learn_df_csv)
print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)
print('Feedback HFA:', feedback_event, feedback_window)
print('Cue HFA:', cue_event, cue_window)
print('N candidate subjects from demographics:', len(subj_ids))


In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}')


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def _extract_corr(res) -> Tuple[float, float]:
    if hasattr(res, 'statistic') and hasattr(res, 'pvalue'):
        return float(res.statistic), float(res.pvalue)
    return float(res[0]), float(res[1])


def run_correlation_analysis(
    x: np.ndarray,
    y: np.ndarray,
    corr_type: str = 'spearman',
    run_secondary_pearson: bool = True,
) -> Dict[str, float]:
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    n = int(len(x_valid))
    if n < 3:
        raise ValueError(f'Need at least 3 valid observations; got {n}.')

    rho_s, p_s = _extract_corr(spearmanr(x_valid, y_valid, nan_policy='omit'))
    if run_secondary_pearson or corr_type.lower() == 'pearson':
        r_p, p_p = _extract_corr(pearsonr(x_valid, y_valid))
    else:
        r_p, p_p = np.nan, np.nan

    corr_type_norm = corr_type.lower()
    if corr_type_norm == 'spearman':
        primary_r, primary_p = rho_s, p_s
    elif corr_type_norm == 'pearson':
        primary_r, primary_p = r_p, p_p
    else:
        raise ValueError(f"Unsupported corr_type: {corr_type}. Use 'spearman' or 'pearson'.")

    return {
        'n': n,
        'spearman_rho': rho_s,
        'spearman_p': p_s,
        'pearson_r': r_p,
        'pearson_p': p_p,
        'primary_corr_type': corr_type_norm,
        'primary_r': primary_r,
        'primary_p': primary_p,
    }


def _fit_and_plot_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    if len(x) < 2 or len(np.unique(x)) < 2:
        return
    coeff = np.polyfit(x, y, 1)
    xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    yfit = np.polyval(coeff, xfit)
    ax.plot(xfit, yfit, color='#222222', linewidth=1.2)


def load_mean_acc_hfa(
    subj_ids: List[str],
    base_dir: str,
    event: str,
    block: Tuple[float, float],
    acc_roi: str = 'ACC',
    hfa_band: str = 'hfa',
) -> pd.DataFrame:
    rows = []
    missing_elec_tables = []
    subjects_with_no_matching_csvs = []
    subjects_with_csvs_but_no_acc_hfa = []
    total_csv_files_found = 0
    block_str = f'time_averaged_df_time{block[0]}-{block[1]}'

    for subj_id in subj_ids:
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df'
        if not os.path.isfile(elec_path):
            missing_elec_tables.append(elec_path)
            continue

        elec_df = pd.read_csv(elec_path)
        region_col = None
        for candidate in ['salman_region', 'region']:
            if candidate in elec_df.columns:
                region_col = candidate
                break
        if region_col is None:
            raise ValueError(f'Could not identify ROI column for {subj_id}.')

        acc_labels = elec_df.loc[elec_df[region_col].astype(str) == acc_roi, 'label'].astype(str).tolist()
        if not acc_labels:
            subjects_with_no_matching_csvs.append(subj_id)
            continue

        subject_parts = []
        found_csv_count = 0

        for chan in acc_labels:
            chan_path = f'{filepath}/{event}/dfs/{chan}_{block_str}.csv'
            if not os.path.isfile(chan_path):
                continue

            found_csv_count += 1
            total_csv_files_found += 1
            chan_df = pd.read_csv(chan_path)
            required = ['tfr', 'fband']
            assert_required_columns(chan_df, required, f'{subj_id} {event} {chan}')

            if 'participant' not in chan_df.columns:
                chan_df['participant'] = subj_id

            if 'region' in chan_df.columns:
                chan_df = chan_df[chan_df['region'].astype(str) == acc_roi]

            chan_df = chan_df[chan_df['fband'].astype(str) == hfa_band]
            if len(chan_df) == 0:
                continue

            chan_df = chan_df[['participant', 'tfr']].copy()
            chan_df['participant'] = subj_id
            subject_parts.append(chan_df)

        if found_csv_count == 0:
            subjects_with_no_matching_csvs.append(subj_id)
            continue
        if not subject_parts:
            subjects_with_csvs_but_no_acc_hfa.append(subj_id)
            continue

        subj_tbl = pd.concat(subject_parts, ignore_index=True)
        rows.append({'Subject': subj_id, 'mean_hfa': float(subj_tbl['tfr'].mean())})

    out = pd.DataFrame(rows)
    if missing_elec_tables:
        print(f'Missing electrode tables for {len(missing_elec_tables)} subject(s).')
    print(f'Loaded {total_csv_files_found} ACC channel CSV(s) for event={event}, block={block}.')
    if subjects_with_no_matching_csvs:
        print(
            'Subjects with no matching ACC CSVs:',
            subjects_with_no_matching_csvs[:10],
            '...' if len(subjects_with_no_matching_csvs) > 10 else ''
        )
    if subjects_with_csvs_but_no_acc_hfa:
        print(
            'Subjects with ACC CSVs but no ACC/HFA rows after filtering:',
            subjects_with_csvs_but_no_acc_hfa[:10],
            '...' if len(subjects_with_csvs_but_no_acc_hfa) > 10 else ''
        )
    if out.empty:
        if total_csv_files_found == 0:
            raise ValueError(
                f'No ACC channel CSV files found for event={event}, block={block}. '
                f'Expected pattern: *_time_averaged_df_time{block[0]}-{block[1]}.csv'
            )
        raise ValueError(
            f'ACC channel CSVs were found for event={event}, block={block}, '
            f'but no rows survived ACC/HFA filtering.'
        )
    return out.sort_values('Subject').reset_index(drop=True)


def plot_subject_covariate_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    predictor_order: List[str],
    predictor_labels: Dict[str, str],
    output_pdf: Path,
    label_points: bool = True,
) -> None:
    n_panels = len(predictor_order)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    sns.set_style('white')

    for ax, predictor in zip(axes, predictor_order):
        panel_df = merged_df[np.isfinite(merged_df[predictor]) & np.isfinite(merged_df[target_col])].copy()
        ax.set_title(predictor_labels[predictor])
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[predictor].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)

        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, predictor), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        rr = results_df[results_df['predictor'] == predictor].iloc[0]
        ann = (
            f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, "
            f"p={rr['primary_p']:.3g}, n={int(rr['n'])}"
        )
        ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)
        ax.set_xlabel(predictor_labels[predictor])
        ax.set_ylabel(target_col)
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    for ax in axes[n_panels:]:
        ax.axis('off')

    fig.suptitle(f'{target_col} vs subject-level ACC covariates', fontsize=14)
    fig.savefig(output_pdf, dpi=300)


In [ ]:
# ---- Load and assemble subject-level analysis table ----
if not os.path.isfile(acc_reinstatement_csv):
    raise FileNotFoundError(f'ACC reinstatement CSV not found: {acc_reinstatement_csv}')
if not os.path.isfile(learn_df_csv):
    raise FileNotFoundError(f'Behavior CSV not found: {learn_df_csv}')

acc_tbl = pd.read_csv(acc_reinstatement_csv)
learn_df = pd.read_csv(learn_df_csv)

assert_required_columns(acc_tbl, ['Subject', reinstatement_metric, 'NumElectrodes'], 'ACC reinstatement table')
assert_required_columns(learn_df, ['participant', 'reward', 'alpha', 'beta'], 'learn_df')

acc_metric_col = f'{acc_roi}_{reinstatement_metric}'
reinstatement_tbl = (
    acc_tbl.rename(columns={
        reinstatement_metric: acc_metric_col,
        'NumElectrodes': 'ACC_NumElectrodes',
    })[['Subject', acc_metric_col, 'ACC_NumElectrodes']]
    .sort_values('Subject')
    .reset_index(drop=True)
)

behavior_consistency_tbl = (
    learn_df.groupby('participant', as_index=False)
    .agg(
        alpha_nunique=('alpha', lambda s: s.dropna().nunique()),
        beta_nunique=('beta', lambda s: s.dropna().nunique()),
    )
    .sort_values('participant')
    .reset_index(drop=True)
)
alpha_conflicts = behavior_consistency_tbl[behavior_consistency_tbl['alpha_nunique'] > 1]
beta_conflicts = behavior_consistency_tbl[behavior_consistency_tbl['beta_nunique'] > 1]
if len(alpha_conflicts) > 0 or len(beta_conflicts) > 0:
    raise ValueError(
        'Found conflicting participant-level alpha/beta values in learn_df. '
        f'alpha_conflicts={alpha_conflicts.to_dict(orient="records")}, '
        f'beta_conflicts={beta_conflicts.to_dict(orient="records")}'
    )
print('Behavior parameter consistency audit:')
display(behavior_consistency_tbl.head())

behavior_tbl = (
    learn_df.groupby('participant', as_index=False)
    .agg(
        total_reward=('reward', lambda s: s.sum(min_count=1)),
        alpha=('alpha', lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan),
        beta=('beta', lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan),
    )
    .rename(columns={'participant': 'Subject'})
    .sort_values('Subject')
    .reset_index(drop=True)
)
print('\nBehavior summary preview:')
display(behavior_tbl.head())

feedback_hfa_tbl = load_mean_acc_hfa(
    subj_ids=subj_ids,
    base_dir=base_dir,
    event=feedback_event,
    block=feedback_window,
    acc_roi=acc_roi,
    hfa_band=hfa_band,
).rename(columns={'mean_hfa': 'ACC_mean_hfa_feedback'})
print('\nFeedback HFA audit:')
print('  n subjects =', feedback_hfa_tbl['Subject'].nunique())
print('  first subjects =', feedback_hfa_tbl['Subject'].head().tolist())
display(feedback_hfa_tbl['ACC_mean_hfa_feedback'].describe().to_frame().T)

cue_hfa_tbl = load_mean_acc_hfa(
    subj_ids=subj_ids,
    base_dir=base_dir,
    event=cue_event,
    block=cue_window,
    acc_roi=acc_roi,
    hfa_band=hfa_band,
).rename(columns={'mean_hfa': 'ACC_mean_hfa_cue'})
print('\nCue HFA audit:')
print('  n subjects =', cue_hfa_tbl['Subject'].nunique())
print('  first subjects =', cue_hfa_tbl['Subject'].head().tolist())
display(cue_hfa_tbl['ACC_mean_hfa_cue'].describe().to_frame().T)

if not os.path.isfile(hpc_theta_reinstatement_csv):
    raise FileNotFoundError(f'HPC theta reinstatement CSV not found: {hpc_theta_reinstatement_csv}')
if not os.path.isfile(acc_hfa_reinstatement_csv):
    raise FileNotFoundError(f'ACC HFA reinstatement CSV not found: {acc_hfa_reinstatement_csv}')

hpc_theta_tbl = pd.read_csv(hpc_theta_reinstatement_csv)
acc_hfa_tbl = pd.read_csv(acc_hfa_reinstatement_csv)
assert_required_columns(hpc_theta_tbl, ['Subject', 'MeanAccuracy', 'NumElectrodes'], 'HPC theta reinstatement table')
assert_required_columns(acc_hfa_tbl, ['Subject', 'MeanAccuracy', 'NumElectrodes'], 'ACC HFA reinstatement table')

hpc_theta_metric_col = 'HPC_theta_MeanAccuracy'
acc_hfa_metric_col = 'ACC_hfa_MeanAccuracy'

hpc_theta_tbl = hpc_theta_tbl.rename(
    columns={
        'MeanAccuracy': hpc_theta_metric_col,
        'NumElectrodes': 'HPC_theta_NumElectrodes',
    }
)[['Subject', hpc_theta_metric_col, 'HPC_theta_NumElectrodes']]

acc_hfa_tbl = acc_hfa_tbl.rename(
    columns={
        'MeanAccuracy': acc_hfa_metric_col,
        'NumElectrodes': 'ACC_hfa_NumElectrodes',
    }
)[['Subject', acc_hfa_metric_col, 'ACC_hfa_NumElectrodes']]

merged_subject_tbl = deterministic_inner_join(reinstatement_tbl, behavior_tbl, keys=['Subject'])
merged_subject_tbl = deterministic_inner_join(merged_subject_tbl, feedback_hfa_tbl, keys=['Subject'])
merged_subject_tbl = deterministic_inner_join(merged_subject_tbl, cue_hfa_tbl, keys=['Subject'])

hpc_merged_subject_tbl = deterministic_inner_join(reinstatement_tbl, hpc_theta_tbl, keys=['Subject'])
hpc_merged_subject_tbl = deterministic_inner_join(hpc_merged_subject_tbl, acc_hfa_tbl, keys=['Subject'])

if len(merged_subject_tbl) == 0:
    raise ValueError('No overlapping subjects after merging reinstatement, reward, and HFA tables.')
if len(hpc_merged_subject_tbl) == 0:
    raise ValueError('No overlapping subjects after merging reinstatement with HPC theta and ACC HFA tables.')

missing_audit = merged_subject_tbl[['Subject', acc_metric_col] + predictor_order].isna().sum()
predictor_counts = []
for predictor in predictor_order:
    n_valid = int(np.isfinite(merged_subject_tbl[predictor].to_numpy(dtype=float)).sum())
    predictor_counts.append({'predictor': predictor, 'n_valid_subjects': n_valid})
    if n_valid < 3:
        raise ValueError(f'Predictor {predictor} has fewer than 3 finite values.')
predictor_counts = pd.DataFrame(predictor_counts)

hpc_missing_audit = hpc_merged_subject_tbl[['Subject', acc_metric_col] + hpc_predictor_order].isna().sum()
hpc_predictor_counts = []
for predictor in hpc_predictor_order:
    n_valid = int(np.isfinite(hpc_merged_subject_tbl[predictor].to_numpy(dtype=float)).sum())
    hpc_predictor_counts.append({'predictor': predictor, 'n_valid_subjects': n_valid})
    if n_valid < 3:
        raise ValueError(f'HPC predictor {predictor} has fewer than 3 finite values.')
hpc_predictor_counts = pd.DataFrame(hpc_predictor_counts)

print('Subjects in ACC reinstatement table:', reinstatement_tbl['Subject'].nunique())
print('Subjects in behavior table:', behavior_tbl['Subject'].nunique())
print('Subjects in feedback HFA table:', feedback_hfa_tbl['Subject'].nunique())
print('Subjects in cue HFA table:', cue_hfa_tbl['Subject'].nunique())
print('Subjects in final merged table:', merged_subject_tbl['Subject'].nunique())
print('Subjects in HPC theta table:', hpc_theta_tbl['Subject'].nunique())
print('Subjects in ACC HFA table:', acc_hfa_tbl['Subject'].nunique())
print('Subjects in HPC theta + ACC HFA merged table:', hpc_merged_subject_tbl['Subject'].nunique())
print('\nPredictor counts:')
display(predictor_counts)
print('\nHPC predictor counts:')
display(hpc_predictor_counts)
print('\nMissing-value audit:')
display(missing_audit.to_frame(name='n_missing').T)
print('\nHPC missing-value audit:')
display(hpc_missing_audit.to_frame(name='n_missing').T)
print('\nMerged table preview:')
display(merged_subject_tbl.head())
print('\nHPC merged table preview:')
display(hpc_merged_subject_tbl.head())


In [ ]:
# ---- Run subject-level correlations ----
result_rows = []

for predictor in predictor_order:
    stats = run_correlation_analysis(
        x=merged_subject_tbl[predictor].to_numpy(dtype=float),
        y=merged_subject_tbl[acc_metric_col].to_numpy(dtype=float),
        corr_type=corr_type,
        run_secondary_pearson=run_secondary_pearson,
    )
    row = {
        'predictor': predictor,
        'predictor_label': predictor_labels[predictor],
        'target_col': acc_metric_col,
        'n': stats['n'],
        'spearman_rho': stats['spearman_rho'],
        'spearman_p': stats['spearman_p'],
        'pearson_r': stats['pearson_r'],
        'pearson_p': stats['pearson_p'],
        'primary_corr_type': stats['primary_corr_type'],
        'primary_r': stats['primary_r'],
        'primary_p': stats['primary_p'],
        'metric_name': reinstatement_metric,
    }
    result_rows.append(row)

results_df = pd.DataFrame(result_rows)
results_df['predictor'] = pd.Categorical(results_df['predictor'], categories=predictor_order, ordered=True)
results_df = results_df.sort_values('predictor').reset_index(drop=True)

if len(results_df) != len(predictor_order):
    raise ValueError(f'Expected {len(predictor_order)} correlation rows; found {len(results_df)}.')

hpc_result_rows = []
for predictor in hpc_predictor_order:
    stats = run_correlation_analysis(
        x=hpc_merged_subject_tbl[predictor].to_numpy(dtype=float),
        y=hpc_merged_subject_tbl[acc_metric_col].to_numpy(dtype=float),
        corr_type=corr_type,
        run_secondary_pearson=run_secondary_pearson,
    )
    row = {
        'predictor': predictor,
        'predictor_label': hpc_predictor_labels[predictor],
        'target_col': acc_metric_col,
        'n': stats['n'],
        'spearman_rho': stats['spearman_rho'],
        'spearman_p': stats['spearman_p'],
        'pearson_r': stats['pearson_r'],
        'pearson_p': stats['pearson_p'],
        'primary_corr_type': stats['primary_corr_type'],
        'primary_r': stats['primary_r'],
        'primary_p': stats['primary_p'],
        'metric_name': reinstatement_metric,
    }
    hpc_result_rows.append(row)

hpc_results_df = pd.DataFrame(hpc_result_rows)
hpc_results_df['predictor'] = pd.Categorical(hpc_results_df['predictor'], categories=hpc_predictor_order, ordered=True)
hpc_results_df = hpc_results_df.sort_values('predictor').reset_index(drop=True)

if len(hpc_results_df) != len(hpc_predictor_order):
    raise ValueError(f'Expected {len(hpc_predictor_order)} HPC correlation rows; found {len(hpc_results_df)}.')

print('Correlation results (uncorrected p-values):')
display(results_df)
print('\nHPC theta + ACC HFA correlation results (uncorrected p-values):')
display(hpc_results_df)


In [ ]:
# ---- Save merged table, correlation table, and text summary ----
merged_subject_tbl.to_csv(paths['merged_csv'], index=False)
results_df.to_csv(paths['results_csv'], index=False)
hpc_merged_subject_tbl.to_csv(paths['hpc_merged_csv'], index=False)
hpc_results_df.to_csv(paths['hpc_results_csv'], index=False)

summary_lines = []
summary_lines.append('ACC REINSTATEMENT VS SUBJECT-LEVEL COVARIATES')
summary_lines.append('===========================================')
summary_lines.append('')
summary_lines.append(f'Reinstatement metric: {reinstatement_metric}')
summary_lines.append(f'Correlation type: {corr_type} (uncorrected p-values)')
summary_lines.append(f'Run secondary Pearson: {run_secondary_pearson}')
summary_lines.append(f'ACC reinstatement file: {acc_reinstatement_csv}')
summary_lines.append(f'HPC theta reinstatement file: {hpc_theta_reinstatement_csv}')
summary_lines.append(f'ACC HFA reinstatement file: {acc_hfa_reinstatement_csv}')
summary_lines.append(f'Behavior file: {learn_df_csv}')
summary_lines.append(f'Feedback HFA event/window: {feedback_event} {feedback_window}')
summary_lines.append(f'Cue HFA event/window: {cue_event} {cue_window}')
summary_lines.append('')
summary_lines.append(f'Subjects in merged table: {merged_subject_tbl["Subject"].nunique()}')
summary_lines.append(f'Subjects in HPC theta + ACC HFA merged table: {hpc_merged_subject_tbl["Subject"].nunique()}')
summary_lines.append('')
summary_lines.append('Predictor counts:')
for row in predictor_counts.itertuples(index=False):
    summary_lines.append(f'  {row.predictor}: n_valid_subjects={row.n_valid_subjects}')
summary_lines.append('')
summary_lines.append('Correlation results:')
for row in results_df.itertuples(index=False):
    summary_lines.append(
        f'  predictor={row.predictor}, n={row.n}, spearman_rho={row.spearman_rho:.6f}, '
        f'spearman_p={row.spearman_p:.6g}, pearson_r={row.pearson_r:.6f}, pearson_p={row.pearson_p:.6g}'
    )
summary_lines.append('')
summary_lines.append('HPC theta + ACC HFA vs ACC reinstatement predictor counts:')
for row in hpc_predictor_counts.itertuples(index=False):
    summary_lines.append(f'  {row.predictor}: n_valid_subjects={row.n_valid_subjects}')
summary_lines.append('')
summary_lines.append('HPC theta + ACC HFA vs ACC reinstatement correlation results:')
for row in hpc_results_df.itertuples(index=False):
    summary_lines.append(
        f'  predictor={row.predictor}, n={row.n}, spearman_rho={row.spearman_rho:.6f}, '
        f'spearman_p={row.spearman_p:.6g}, pearson_r={row.pearson_r:.6f}, pearson_p={row.pearson_p:.6g}'
    )

with open(paths['summary_txt'], 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines) + '\n')

print('Saved merged analysis table:', paths['merged_csv'])
print('Saved correlation results:', paths['results_csv'])
print('Saved HPC merged analysis table:', paths['hpc_merged_csv'])
print('Saved HPC correlation results:', paths['hpc_results_csv'])
print('Saved summary text:', paths['summary_txt'])


In [ ]:
# ---- Save scatter grid ----
plot_subject_covariate_grid(
    merged_df=merged_subject_tbl,
    results_df=results_df,
    target_col=acc_metric_col,
    predictor_order=predictor_order,
    predictor_labels=predictor_labels,
    output_pdf=paths['scatter_pdf'],
    label_points=label_points,
)

print('Saved scatter plot:', paths['scatter_pdf'])


In [ ]:
# ---- Save HPC theta + ACC HFA vs ACC reinstatement scatter grid ----
plot_subject_covariate_grid(
    merged_df=hpc_merged_subject_tbl,
    results_df=hpc_results_df,
    target_col=acc_metric_col,
    predictor_order=hpc_predictor_order,
    predictor_labels=hpc_predictor_labels,
    output_pdf=paths['hpc_scatter_pdf'],
    label_points=label_points,
)

print('Saved HPC theta + ACC HFA scatter plot:', paths['hpc_scatter_pdf'])
